## Import libraries


In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = "Times New Roman"

## Read the data


In [ ]:
ROOT = Path("/beegfs/halder/GITHUB/RESEARCH/WBCrop")
pts_path = ROOT / "data" / "master" / "wbcrop_points_final.gpkg"
fields_path = ROOT / "data" / "master" / "wbcrop_fields.gpkg"
wb_path = ROOT / "data" / "raw" / "shapefile" / "west_bengal.gpkg"

# load data
pts = gpd.read_file(pts_path)
fields = gpd.read_file(fields_path)
wb = gpd.read_file(wb_path)

# Ensure everything is in WGS 84 for the spatial map
pts_wgs = pts.to_crs(epsg=4326)
fields_wgs = fields.to_crs(epsg=4326)
wb_wgs = wb.to_crs(epsg=4326)

print(f"Points : {len(pts_wgs):,}  |  Fields : {len(fields_wgs):,}")
print(f"Crop classes (points) : {sorted(pts_wgs['crop'].unique())}")
print(f"Crop classes (fields) : {sorted(fields_wgs['crop'].unique())}")

## Class Distribution (18 Crop Classes) for Points & Fields


In [ ]:
# crop label map
LABEL_MAP = {
    "aman_rice": "Aman Rice",
    "aus_rice": "Aus Rice",
    "banana": "Banana",
    "betel_leaf": "Betel Leaf",
    "boro_rice": "Boro Rice",
    "flower": "Flower",
    "groundnut": "Groundnut",
    "jute": "Jute",
    "maize": "Maize",
    "mustard": "Mustard",
    "others": "Others",
    "pine_apple": "Pineapple",
    "potato": "Potato",
    "sugarcane": "Sugarcane",
    "tea": "Tea",
    "tobacco": "Tobacco",
    "vegetables": "Vegetables",
    "wheat": "Wheat",
}


# 18-class qualitative palette (tab20 minus 2 duplicates)
cmap = plt.cm.get_cmap("tab20", 20)
CROP_ORDER = sorted(LABEL_MAP.keys())
COLOR_MAP = {c: cmap(i) for i, c in enumerate(CROP_ORDER)}

# season colour map
SEASON_MAP = {
    "aman_rice": "Kharif",
    "aus_rice": "Zaid",
    "banana": "Perennial",
    "betel_leaf": "Perennial",
    "boro_rice": "Rabi",
    "flower": "Rabi",
    "groundnut": "Zaid",
    "jute": "Kharif",
    "maize": "Rabi",
    "mustard": "Rabi",
    "others": "Rabi",
    "pine_apple": "Perennial",
    "potato": "Rabi",
    "sugarcane": "Perennial",
    "tea": "Perennial",
    "tobacco": "Rabi",
    "vegetables": "Rabi",
    "wheat": "Rabi",
}

SEASON_COLORS = {
    "Kharif": "#58B368",
    "Rabi": "#FF7657",
    "Zaid": "#FFBA5A",
    "Perennial": "#665C84",
}

In [ ]:
# Prepare data for plotting class distribution
pts_counts = pts_wgs["crop"].value_counts().reindex(CROP_ORDER)
labels = [LABEL_MAP[crop] for crop in CROP_ORDER]
x = np.arange(len(CROP_ORDER))
bar_w = 0.35

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

bar_colors = [SEASON_COLORS[SEASON_MAP[crop]] for crop in CROP_ORDER]

bars1 = ax.bar(
    x,
    pts_counts.values,
    bar_w * 2,
    label="Points",
    color=bar_colors,
    edgecolor="k",
    linewidth=1,
    alpha=0.85,
)

# count labels above bars
for bar in bars1:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f"{int(bar.get_height()):,}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        rotation=90,
    )

# imbalance ratio line
ratio = pts_counts.max() / pts_counts.min()

# axes formatting
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=10)
ax.set_ylabel("Number of Samples", fontsize=12)
ax.set_xlim(-0.6, len(CROP_ORDER) - 0.4)
ax.set_ylim(0, pts_counts.max() * 1.20)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))

from matplotlib.lines import Line2D

season_handles = [
    mpatches.Patch(facecolor=SEASON_COLORS[s], label=s, edgecolor="k")
    for s in ["Kharif", "Rabi", "Zaid", "Perennial"]
]
ax.legend(
    handles=season_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=12,
    ncol=6,
    frameon=False,
    fancybox=True,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
# plt.savefig(
#     ROOT / "output" / "figures" / "class_distribution_points.png",
#     dpi=300,
#     bbox_inches="tight",
#     facecolor="white",
# )
plt.show()

## Crop Distribution per District


In [ ]:
# Styled crop-distribution table (district × crop)
import pandas as pd

#  Build the pivot: rows = district (Title Case), cols = crop (pretty)
pivot = pts_wgs.groupby(["district", "crop"]).size().unstack(fill_value=0)
# Pretty column names
pivot.columns = [LABEL_MAP[c] for c in pivot.columns]
# Pretty row names (Title Case)
dist_name_map = dict(
    zip(
        wb_wgs["Name"].str.lower().str.strip(),
        wb_wgs["Name"],
    )
)
pivot.index = pivot.index.map(lambda d: dist_name_map.get(d, d.title()))
pivot.index.name = "District"

# Add row totals
pivot["Total"] = pivot.sum(axis=1)

# Add column totals row
col_totals = pivot.sum(axis=0)
col_totals.name = "Total"
pivot = pd.concat([pivot, col_totals.to_frame().T])

# Sort rows: districts alphabetically, Total at bottom
district_rows = pivot.iloc[:-1].sort_index()
pivot = pd.concat([district_rows, pivot.iloc[[-1]]])


# Style the table
def style_table(styler):
    """Apply a publication-ready style to the pivot table."""

    # Colour gradient on the crop columns (exclude 'Total')
    crop_cols = [c for c in styler.columns if c != "Total"]

    styler = (
        styler
        # Gradient on crop cells (row-wise so colour is relative within each district)
        .background_gradient(
            cmap="YlOrRd", subset=(styler.index[:-1], crop_cols), axis=1
        )
        # Bold total column
        .background_gradient(
            cmap="Blues", subset=(styler.index[:-1], ["Total"]), axis=0
        )
        # Highlight the grand-total row
        .set_properties(
            subset=(pd.IndexSlice[styler.index[-1]], styler.columns),
            **{"font-weight": "bold", "background-color": "#e8e8e8"},
        )
        # Format numbers with comma separator
        .format("{:,.0f}")
        # General styling
        .set_table_styles(
            [
                # Header row
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#4a4a4a"),
                        ("color", "white"),
                        ("font-size", "11px"),
                        ("text-align", "center"),
                        ("padding", "6px 8px"),
                        ("border", "1px solid #ddd"),
                    ],
                },
                # Index (district names)
                {
                    "selector": "th.row_heading",
                    "props": [
                        ("background-color", "#5a5a5a"),
                        ("color", "white"),
                        ("font-size", "11px"),
                        ("text-align", "left"),
                        ("padding", "6px 10px"),
                        ("white-space", "nowrap"),
                    ],
                },
                # Data cells
                {
                    "selector": "td",
                    "props": [
                        ("font-size", "18px"),
                        ("font-weight", "bold"),
                        ("text-align", "center"),
                        ("padding", "5px 7px"),
                        ("border", "1px solid black"),
                    ],
                },
                # Caption
                {
                    "selector": "caption",
                    "props": [
                        ("caption-side", "top"),
                        ("font-size", "13px"),
                        ("font-weight", "bold"),
                        ("padding-bottom", "8px"),
                        ("color", "#333"),
                    ],
                },
            ]
        ).set_caption("Crop Sample Distribution per District (Point Samples)")
        # Highlight zeros so they're visually muted
        .map(lambda v: "color: #ccc;" if v == 0 else "", subset=crop_cols)
    )
    return styler


styled = pivot.style.pipe(style_table)
styled.to_excel(
    ROOT / "output" / "tables" / "crop_distribution_per_district.xlsx",
    engine="openpyxl",
)

import dataframe_image as dfi

dfi.export(
    styled,
    ROOT / "output" / "figures" / "crop_distribution_heatmap.png",
    dpi=300,
    table_conversion="matplotlib",
)